<a href="https://colab.research.google.com/github/Fanny0504/Lab1/blob/main/Practica_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Parte1

In [ ]:
import re
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg')

def clean_text(text):
    text = text.lower()  # Case folding
    # Mantener solo letras y espacios
    text = re.sub(r'[^a-z\s]', '', text)
    # Eliminar espacios múltiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw_text = gutenberg.raw('shakespeare-hamlet.txt')
cleaned_text = clean_text(raw_text)
print(cleaned_text[:200])  # Vista previa


the tragedie of hamlet by william shakespeare actus primus scoena prima enter barnardo and francisco two centinels barnardo whos there fran nay answer me stand vnfold your selfe bar long liue the king


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


Parte 2

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import spacy

nlp = spacy.load("en_core_web_sm")

tokens = word_tokenize(cleaned_text)

# A. Stemming
stemmer = PorterStemmer()
stems = [stemmer.stem(t) for t in tokens]

# B. Lematización
doc = nlp(" ".join(tokens[:1000]))  # Muestra para no saturar
lemmas = [t.lemma_ for t in doc]

# Comparación
print("Original:", tokens[:20])
print("Stems:", stems[:20])
print("Lemmas:", lemmas[:20])


Original: ['the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', 'barnardo', 'whos', 'there']
Stems: ['the', 'tragedi', 'of', 'hamlet', 'by', 'william', 'shakespear', 'actu', 'primu', 'scoena', 'prima', 'enter', 'barnardo', 'and', 'francisco', 'two', 'centinel', 'barnardo', 'who', 'there']
Lemmas: ['the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', 'actus', 'primus', 'scoena', 'prima', 'enter', 'barnardo', 'and', 'francisco', 'two', 'centinel', 'barnardo', 'who', 's']


Parte 3

In [ ]:
import numpy as np
import nltk # Solo para comparar al final como pide la práctica

def min_edit_distance(source, target):
    n = len(source)
    m = len(target)

    # 1. Crear matriz de ceros de tamaño (n+1)x(m+1)
    D = np.zeros((n + 1, m + 1), dtype=int)

    # 2. Llenar primera fila y columna con los costos base
    for i in range(n + 1):
        D[i, 0] = i
    for j in range(m + 1):
        D[0, j] = j

    # 3. Recurrencia (Programación Dinámica)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            # Costo de sustitución: 0 si coinciden, 2 si son diferentes
            cost = 0 if source[i-1] == target[j-1] else 2

            # Aplicar la fórmula: min(borrado, inserción, sustitución)
            D[i, j] = min(
                D[i-1, j] + 1,      # Borrado
                D[i, j-1] + 1,      # Inserción
                D[i-1, j-1] + cost  # Sustitución
            )

    # 4. Retornar el valor de la celda inferior derecha
    return D[n, m]

# --- EJECUCIÓN DE PRUEBAS  ---
pares_prueba = [("gato", "pato"), ("intention", "execution"), ("learning", "leaning")]

print(f"{'Palabras':<30} | {'Manual (Costo 2)':<18} | {'NLTK (Costo 1)':<15}")
print("-" * 70)

for s, t in pares_prueba:
    dist_manual = min_edit_distance(s, t)
    # NLTK usa por defecto costo 1 para sustitución
    dist_nltk = nltk.edit_distance(s, t, substitution_cost=1)

    print(f"{str((s,t)):<30} | {dist_manual:<18} | {dist_nltk:<15}")


Palabras                       | Manual (Costo 2)   | NLTK (Costo 1) 
----------------------------------------------------------------------
('gato', 'pato')               | 2                  | 1              
('intention', 'execution')     | 8                  | 5              
('learning', 'leaning')        | 1                  | 1              


Parte 4


In [ ]:
def simple_spell_checker(word, vocabulary):
    # Generar un set para eliminar duplicados y que la búsqueda sea más rápida
    unique_vocab = list(set(vocabulary))

    palabra_sugerida = None
    distancia_minima = float('inf')

    # Iterar sobre el vocabulario para encontrar la menor distancia de edición
    for vocab_word in unique_vocab:
        # Usamos la función de la Parte 3 con costo de sustitución = 2
        distancia_actual = min_edit_distance(word, vocab_word)

        if distancia_actual < distancia_minima:
            distancia_minima = distancia_actual
            palabra_sugerida = vocab_word

        # Si la distancia es 0, encontramos la palabra exacta y terminamos el bucle
        if distancia_minima == 0:
            break

    return palabra_sugerida, distancia_minima

# --- EJECUCIÓN PARA OBTENER RESULTADOS  ---
# Usamos una palabra mal escrita y comparamos contra los tokens de la Parte 2
palabra_usuario = "tragedi" # Intento de escribir 'tragedie'
sugerencia, d = simple_spell_checker(palabra_usuario, tokens[:1000])

print(f"Palabra ingresada: {palabra_usuario}")
print(f"Sugerencia más cercana: {sugerencia}")
print(f"Distancia de edición calculada: {d}")

Palabra ingresada: tragedi
Sugerencia más cercana: tragedie
Distancia de edición calculada: 1
